# 优化字幕生成器演示

本 notebook 演示如何使用优化的字幕拼接方案生成电影字幕。

## 1. 基础使用

In [ ]:
from generate_subtitles import MovieSubtitleGenerator, SubtitleConfig, quick_generate

# 方式1: 使用配置对象
config = SubtitleConfig(
    asr_model_path='/mnt/g/models/Qwen3-ASR-1.7B',
    aligner_path='/mnt/g/models/Qwen3-ForcedAligner-0.6B',
    language='Chinese',  # 或 None 自动检测，或 'English'
    max_chars=40,
    max_duration=6.0,
)

generator = MovieSubtitleGenerator(config)

# 生成字幕
srt_path = generator.generate(
    video_path='/path/to/movie.mp4',
    output_srt='/path/to/output.srt',
    save_intermediate=True  # 保存中间结果（VAD分割、词对齐）
)

## 2. 快速使用

In [ ]:
from generate_subtitles import quick_generate

# 一行代码生成字幕
srt_path = quick_generate(
    video_path='/mnt/mininas/4t2/new/savr-401/4k2.com@savr-401_1_8k.mp4',
    language='Chinese',
)

## 3. 分步处理（使用已有的 VAD 片段）

In [ ]:
import torch
import torchaudio
from subtitle_merger import SubtitleMerger, load_segments_json, save_subtitles

# 3.1 加载已有的 VAD 片段信息
segments = load_segments_json('segments/segments.json')
print(f"加载了 {len(segments)} 个片段")

# 3.2 初始化字幕合并器（可以调整参数）
merger = SubtitleMerger(
    max_chars=40,           # 每行最大字符数
    max_duration=6.0,       # 每行最大显示时长
    min_duration=1.0,       # 每行最小显示时长
    gap_threshold=0.3,      # 合并相邻片段的间隔阈值
    target_reading_speed=6.0,  # 目标阅读速度（字/秒）
)

# 3.3 模拟 ASR + ForcedAligner 结果
# 实际使用时，这里应该是 ASR 转录的词级时间戳
from subtitle_merger import WordItem

# 示例数据（实际使用时应从 ASR 获取）
mock_words = []
for seg in segments[:5]:  # 只处理前5个片段作为示例
    # 模拟每个片段有若干个词
    words = [
        WordItem("这", seg['start'] + 0.1, seg['start'] + 0.3),
        WordItem("是", seg['start'] + 0.3, seg['start'] + 0.5),
        WordItem("测", seg['start'] + 0.5, seg['start'] + 0.7),
        WordItem("试", seg['start'] + 0.7, seg['start'] + 0.9),
    ]
    mock_words.extend(words)

# 3.4 处理字幕
subtitle_lines = merger.process(segments[:5], mock_words)

# 3.5 保存字幕
save_subtitles(subtitle_lines, 'output_demo.srt')

## 4. 单独使用 VAD + 字幕优化（无 ASR）

In [ ]:
import torch
import torchaudio

# 4.1 加载音频
wav_path = '4k2.com@savr-401_1_8k.wav'
wav, sample_rate = torchaudio.load(wav_path)
wav = wav.squeeze()

# 4.2 VAD 分割
model, utils = torch.hub.load('snakers4/silero-vad', model='silero_vad')
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

speech_timestamps = get_speech_timestamps(
    wav, model, sampling_rate=16000,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=500,
)

segments = [
    {'index': i, 'start': ts['start']/16000, 'end': ts['end']/16000}
    for i, ts in enumerate(speech_timestamps)
]

print(f"检测到 {len(segments)} 个语音片段")

# 4.3 查看 VAD 合并效果
merger = SubtitleMerger(gap_threshold=0.3)
merged_segments = merger.merge_vad_segments(segments)

print(f"\n合并后: {len(merged_segments)} 个片段")
print(f"减少: {len(segments) - len(merged_segments)} 个片段")

## 5. 高级参数调优示例

In [ ]:
# 不同场景的参数配置

# 配置 A: 电影（对话密集，语速正常）
movie_config = SubtitleConfig(
    language='Chinese',
    max_chars=40,
    max_duration=6.0,
    gap_threshold=0.3,  # 合并短间隔
    vad_min_silence_duration_ms=300,  # 较短的分段间隔
)

# 配置 B: 纪录片（旁白为主，语速较慢）
documentary_config = SubtitleConfig(
    language='Chinese',
    max_chars=35,       # 稍短，方便阅读
    max_duration=7.0,   # 可以显示更久
    gap_threshold=0.5,  # 不太合并
    vad_min_silence_duration_ms=800,  # 较长停顿才分段
)

# 配置 C: 动画（语速快，断句紧凑）
anime_config = SubtitleConfig(
    language='Japanese',
    max_chars=30,       # 较短
    max_duration=4.0,   # 显示时间短
    gap_threshold=0.2,  # 积极合并
    target_reading_speed=8.0,  # 适应快语速
)

## 6. 查看生成的字幕效果

In [ ]:
# 读取并显示生成的字幕
def preview_srt(srt_path, n=10):
    """预览 SRT 文件前 n 行"""
    with open(srt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    blocks = content.strip().split('\n\n')
    print(f"总字幕行数: {len(blocks)}\n")
    
    for block in blocks[:n]:
        print(block)
        print()

# 使用示例
# preview_srt('output.srt', n=5)

## 7. 命令行使用

In [ ]:
# 在终端中使用
# python generate_subtitles.py /path/to/movie.mp4 -o output.srt -l Chinese

# 查看帮助
# python generate_subtitles.py --help